# Small Dynamic Factor Model: Factor Extraction and Diagnostics

**Module 04 — Notebook 01**  
**Estimated time:** 15 minutes

## Learning Objectives

1. Build a balanced monthly panel from GDP-related indicators
2. Extract common factors via PCA and select the number of factors
3. Interpret factor loadings economically
4. Visualize the first factor alongside GDP growth (correlation check)
5. Apply Bai-Ng criterion for factor count selection

## Setup

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Libraries loaded.")

In [ ]:
section_divider("1. Load Monthly Panel")

In [ ]:
learning_objectives(["Build a balanced monthly panel from GDP-related indicators", "Extract common factors via PCA and select the number of factors", "Interpret factor loadings economically", "Visualize the first factor alongside GDP growth (correlation check)", "Apply Bai-Ng criterion for factor count selection"])

In [ ]:
apply_course_theme()
apply_plot_theme()

## 1. Load Monthly Panel

We use IP growth plus two synthetic correlated indicators to form a 3-indicator panel. For real applications, add PAYEMS (payrolls) and retail sales from FRED.

In [ ]:
USE_FRED = False

def load_series_from_csv(series_id):
    import os
    candidates = [
        '../resources',
        '../../module_00_foundations/resources',
        '../../../module_00_foundations/resources',
        '../../../../module_00_foundations/resources',
    ]
    filename_map = {
        'GDPC1':  'gdp_quarterly.csv',
        'INDPRO': 'industrial_production_monthly.csv',
        'SP500':  'sp500_daily.csv',
    }
    fname = filename_map[series_id]
    for base in candidates:
        path = os.path.join(base, fname)
        if os.path.exists(path):
            df = pd.read_csv(path, index_col='date', parse_dates=True)
            return df.squeeze()
    raise FileNotFoundError(f"Cannot find {fname}")


if USE_FRED:
    from fredapi import Fred
    fred = Fred()
    gdp_raw  = fred.get_series('GDPC1',  observation_start='2000-01-01')
    ip_raw   = fred.get_series('INDPRO', observation_start='1999-01-01')
    pay_raw  = fred.get_series('PAYEMS', observation_start='1999-01-01')
    gdp_growth = np.log(gdp_raw).diff().dropna() * 100
    ip_growth  = np.log(ip_raw).diff().dropna() * 100
    pay_growth = np.log(pay_raw).diff().dropna() * 100
    indicators = {'IP': ip_growth, 'Payrolls': pay_growth}
else:
    gdp_growth = load_series_from_csv('GDPC1')
    ip_growth  = load_series_from_csv('INDPRO')

    # Build synthetic panel: IP + two correlated noisy variants
    # In practice: replace with PAYEMS and retail sales from FRED
    np.random.seed(10)
    ip_vals = ip_growth.values
    n_months = len(ip_vals)

    # Synthetic payrolls: 0.7 * IP + noise (correlated with business cycle)
    payrolls_syn = 0.7 * ip_vals + 0.5 * np.random.randn(n_months)
    # Synthetic retail sales: 0.6 * IP + noise
    retail_syn = 0.6 * ip_vals + 0.6 * np.random.randn(n_months)

    indicators = {
        'IP':       pd.Series(ip_vals, index=ip_growth.index, name='IP'),
        'Payrolls': pd.Series(payrolls_syn, index=ip_growth.index, name='Payrolls'),
        'Retail':   pd.Series(retail_syn, index=ip_growth.index, name='Retail'),
    }

# Build balanced panel (drop any row with NaN)
panel_raw = pd.DataFrame(indicators)
panel_raw = panel_raw.dropna()

print(f"Panel: {panel_raw.shape[1]} indicators, {len(panel_raw)} monthly observations")
print(f"Date range: {panel_raw.index[0].date()} — {panel_raw.index[-1].date()}")
print(f"GDP: {len(gdp_growth)} quarterly observations")

In [ ]:
section_divider("2. Standardize and Extract Factors")

## 2. Standardize and Extract Factors

In [ ]:
# Standardize
scaler = StandardScaler()
X_std = scaler.fit_transform(panel_raw.values)
X_std_df = pd.DataFrame(X_std, index=panel_raw.index, columns=panel_raw.columns)

print("Descriptive statistics (standardized):")
print(X_std_df.describe().round(3))
print("\nCorrelation matrix:")
print(X_std_df.corr().round(4))

In [ ]:
def extract_factors_pca(X_std, n_factors, indicator_names, reference_col=0):
    """
    Extract n_factors common factors via PCA.
    Normalizes sign so reference indicator has positive loading.
    """
    pca = PCA(n_components=n_factors)
    F = pca.fit_transform(X_std)           # (T, n_factors)
    Lambda = pca.components_.T             # (N, n_factors)

    # Sign normalization: reference indicator (IP) loading must be positive
    for j in range(n_factors):
        if Lambda[reference_col, j] < 0:
            F[:, j] = -F[:, j]
            Lambda[:, j] = -Lambda[:, j]

    var_exp = pca.explained_variance_ratio_

    print(f"Factor extraction: q={n_factors}")
    print(f"Variance explained: {', '.join(f'{v:.1%}' for v in var_exp)}")
    print(f"Total: {var_exp.sum():.1%}")
    print("\nFactor loadings:")
    for i, name in enumerate(indicator_names):
        loadings_str = ', '.join(f'F{j+1}={Lambda[i,j]:+.4f}' for j in range(n_factors))
        print(f"  {name:<12}: {loadings_str}")

    return F, Lambda, var_exp


# Extract factors
N_FACTORS = 2
F, Lambda, var_exp = extract_factors_pca(
    X_std, N_FACTORS, list(panel_raw.columns), reference_col=0
)

F_df = pd.DataFrame(F, index=panel_raw.index,
                    columns=[f'Factor{j+1}' for j in range(N_FACTORS)])

In [ ]:
section_divider("3. Bai-Ng Factor Count Selection")

## 3. Bai-Ng Factor Count Selection

In [ ]:
def bai_ng_criterion(X_std, max_factors=None):
    """
    Bai-Ng IC_p2 criterion for number of factors.
    Returns optimal q and all IC values.
    """
    T, N = X_std.shape
    if max_factors is None:
        max_factors = min(N - 1, 8)
    g_NT = (N + T) / (N * T) * np.log(min(N, T))

    ic_values = []
    for q in range(1, max_factors + 1):
        pca = PCA(n_components=q)
        F_q = pca.fit_transform(X_std)
        Lambda_q = pca.components_.T
        X_hat = F_q @ Lambda_q.T
        sigma2 = np.mean((X_std - X_hat)**2)
        ic = np.log(sigma2) + q * g_NT
        ic_values.append(ic)

    q_opt = np.argmin(ic_values) + 1

    print(f"Bai-Ng IC_p2: g(N={N_std},T={T_std}) = {g_NT:.6f}")
    print(f"\nq   IC_p2")
    print("-" * 20)
    for q, ic in enumerate(ic_values, 1):
        marker = " <- optimal" if q == q_opt else ""
        print(f"{q:3d}  {ic:10.6f}{marker}")
    print(f"\nOptimal number of factors: q={q_opt}")
    return q_opt, np.array(ic_values)


T_std, N_std = X_std.shape
q_opt, ic_vals = bai_ng_criterion(X_std)

In [ ]:
section_divider("4. Visualization")

## 4. Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Panel 1: Scree plot
ax = axes[0, 0]
q_range = np.arange(1, len(ic_vals) + 1)
ax.bar(q_range, ic_vals - ic_vals.min(), color='steelblue', alpha=0.8)
ax.axvline(q_opt, color='tomato', linewidth=2.5, linestyle='--', label=f'Optimal q={q_opt}')
ax.set_xlabel('Number of factors q')
ax.set_ylabel('Bai-Ng IC_p2 (relative to minimum)')
ax.set_title('Bai-Ng Criterion for Factor Count')
ax.legend()
ax.set_xticks(q_range)

# Panel 2: Variance explained
all_var = []
for q in range(1, min(N_std, 6) + 1):
    pca_q = PCA(n_components=q)
    pca_q.fit(X_std)
    all_var.append(pca_q.explained_variance_ratio_[q-1])

ax2 = axes[0, 1]
q_arr = np.arange(1, len(all_var) + 1)
ax2.bar(q_arr, np.cumsum(all_var), alpha=0.5, color='steelblue', label='Cumulative')
ax2.bar(q_arr, all_var, alpha=0.85, color='tomato', label='Marginal')
ax2.axhline(0.5, color='gray', linestyle='--', linewidth=1, label='50% threshold')
ax2.set_xlabel('Number of factors')
ax2.set_ylabel('Variance explained')
ax2.set_title('Variance Explained by Factors')
ax2.legend(fontsize=9)
ax2.set_xticks(q_arr)

# Panel 3: Factor 1 vs. quarterly aggregated (aligned with GDP)
ax3 = axes[1, 0]
# Aggregate F1 to quarterly
F1_series = pd.Series(F[:, 0], index=panel_raw.index)
if hasattr(F1_series.index, 'to_period'):
    F1_q = F1_series.groupby(F1_series.index.to_period('Q')).mean()
else:
    F1_series.index = pd.to_datetime(F1_series.index)
    F1_q = F1_series.resample('QE').mean()

# Align with GDP
if hasattr(gdp_growth.index, 'to_period'):
    gdp_q = gdp_growth.copy()
else:
    gdp_q = gdp_growth.copy()

common_idx = F1_q.index[:len(gdp_q)]
n_common = min(len(F1_q), len(gdp_q))
f1_arr = F1_q.values[:n_common]
gdp_arr = gdp_q.values[:n_common]

# Normalize F1 to same scale as GDP for visual comparison
f1_norm = (f1_arr - f1_arr.mean()) / f1_arr.std() * gdp_arr.std() + gdp_arr.mean()

ax3.plot(range(n_common), gdp_arr, color='steelblue', linewidth=1.5,
         label='Actual GDP growth')
ax3.plot(range(n_common), f1_norm, color='tomato', linewidth=1.5, linestyle='--',
         label='Factor 1 (rescaled)')
ax3.axhline(0, color='gray', linewidth=0.7, alpha=0.5)
corr = np.corrcoef(f1_arr, gdp_arr)[0, 1]
ax3.set_title(f'GDP vs. Factor 1 (r = {corr:.3f})')
ax3.set_xlabel('Quarter')
ax3.set_ylabel('Growth rate (%)')
ax3.legend(fontsize=9)

# Panel 4: Factor loadings bar chart
ax4 = axes[1, 1]
indicator_names = list(panel_raw.columns)
x_pos = np.arange(len(indicator_names))
width = 0.35
n_show = min(2, Lambda.shape[1])
colors = ['steelblue', 'tomato', 'green']
for j in range(n_show):
    ax4.bar(x_pos + j * width - width/2 * (n_show-1) / n_show,
            Lambda[:, j], width, alpha=0.85,
            color=colors[j], label=f'Factor {j+1}')
ax4.axhline(0, color='black', linewidth=0.8)
ax4.set_xticks(x_pos)
ax4.set_xticklabels(indicator_names, fontsize=10)
ax4.set_ylabel('Loading')
ax4.set_title('Factor Loadings')
ax4.legend(fontsize=9)

plt.suptitle('Small DFM: Factor Extraction Diagnostics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('small_dfm_diagnostics.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Figure saved. Correlation of F1 with GDP: {corr:.4f}")

In [ ]:
section_divider("5. Self-Check")

## 5. Self-Check

In [ ]:
print("Self-check...")
passed = 0
total = 5

# 1. Standardized panel has unit variance per column
col_stds = X_std.std(axis=0)
assert all(abs(s - 1.0) < 0.01 for s in col_stds), \
    f"FAIL 1: Standardized columns should have std≈1, got {col_stds}"
passed += 1
print(f"  PASS 1: All columns standardized to unit variance")

# 2. Factor loadings have correct shape
assert Lambda.shape == (N_std, N_FACTORS), \
    f"FAIL 2: Lambda shape {Lambda.shape} != ({N_std}, {N_FACTORS})"
passed += 1
print(f"  PASS 2: Lambda shape = {Lambda.shape}")

# 3. IP loading (col 0) on Factor 1 is positive (sign normalization)
assert Lambda[0, 0] > 0, \
    f"FAIL 3: IP loading on F1 ({Lambda[0,0]:.4f}) should be positive after normalization"
passed += 1
print(f"  PASS 3: IP loading on F1 = {Lambda[0,0]:.4f} (positive)")

# 4. Variance explained by all factors sums to <= 1
assert var_exp.sum() <= 1.001, \
    f"FAIL 4: Total variance explained ({var_exp.sum():.4f}) must be <= 1"
passed += 1
print(f"  PASS 4: Total variance explained = {var_exp.sum():.4f} <= 1")

# 5. Factor 1 is correlated with GDP (|r| > 0.3 for a reasonable dataset)
assert abs(corr) > 0.3, \
    f"FAIL 5: |Corr(F1, GDP)| = {abs(corr):.4f} should be > 0.3"
passed += 1
print(f"  PASS 5: |Corr(F1, GDP)| = {abs(corr):.4f} > 0.3")

print(f"\n{'='*40}")
print(f"Self-check: {passed}/{total} passed")
if passed == total:
    print("All checks passed.")

## Summary

| Output | Value |
|--------|-------|
| N indicators | 3 |
| T monthly obs | ~300 |
| Optimal q (Bai-Ng) | computed above |
| F1 variance explained | first element of var_exp |
| F1–GDP correlation | computed above |

### Next

**Notebook 02:** Factor-augmented MIDAS — use the extracted factor as the MIDAS predictor and compare RMSE to single-indicator MIDAS.

In [ ]:
key_takeaways(["Factor-augmented MIDAS — use the extracted factor as the MIDAS predictor and com"])